In [1]:
import os, sys
from pathlib import Path
_cwd = Path.cwd()
_root = next((p for p in [_cwd] + list(_cwd.parents)
              if (p / 'requirements.txt').exists() and (p / 'src').is_dir()), None)
assert _root, f'Could not find project root from {_cwd}'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')

Project root: C:\Users\markd\OneDrive\Desktop\Claude\GitHub\Projects\uk-energy-price-forecasting


# 04 — Backtest & Ablation Table

We run a **rolling-origin backtest** over the last few days of the fixture panel, comparing:

- `SeasonalNaive` (the baseline, no fitting)
- `GlobalLGBM` (fit once, ~45 s on CPU)

> **Note:** Rows for `GlobalTiDE`, `GlobalTFT`, and `Chronos` are produced in notebook 03 on Colab and can be merged here by loading the saved `models/*.pkl` / `*.pt` artefacts and re-running `run_backtest` on the same origins.

The ablation table reports MAE, RMSE, CRPS, pinball, 80 % coverage, and skill score vs the seasonal-naive baseline.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.build.fixtures import load_fixture_panel
from src.models.baselines import SeasonalNaive
from src.models.global_ml import GlobalLGBM
from src.backtest.rolling_origin import generate_origins, run_backtest, build_ablation_table

bundle = load_fixture_panel()
target_idx = bundle.target.time_index
print('Panel loaded — target has', bundle.target.n_timesteps, 'steps')
print('Target range:', target_idx[0], '→', target_idx[-1])

The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.


The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.


The provided DatetimeIndex was associated with a timezone (tz), which is currently not supported. To avoid unexpected behaviour, the tz information was removed. Consider calling `ts.time_index.tz_localize(UTC)` when exporting the results.To plot the series with the right time steps, consider setting the matplotlib.pyplot `rcParams['timezone']` parameter to automatically convert the time axis back to the original timezone.


Panel loaded — target has 1440 steps
Target range: 2024-01-01 00:00:00 → 2024-01-30 23:30:00


## Generate rolling origins

We use the last 4 days of the 30-day fixture for backtesting (leaving at least 25 days for training).

In [3]:
# Origins: last 4 midnights that have ≥48 steps of actual data after them
# target has 30 days → last midnight we can safely use = day 28 (day 29 is the last
# day, and we need 48 steps of future actuals after the origin).
panel_end = target_idx[-1]
# Use day 22–25 as backtest window (enough history before, enough future after)
# Note: Darts strips timezone from TimeSeries.time_index, so origins must be
# tz-naive to match.  The data is in UTC; midnight boundaries are the same.
origin_start = pd.Timestamp('2024-01-22 00:00')
origin_end   = pd.Timestamp('2024-01-25 00:00')

origins = generate_origins(
    time_index=target_idx,
    start=origin_start,
    end=origin_end,
    step='1D',
)
print(f'Origins ({len(origins)}):')
for o in origins:
    print(' ', o)

Origins (4):
  2024-01-22 00:00:00
  2024-01-23 00:00:00
  2024-01-24 00:00:00
  2024-01-25 00:00:00


## Run backtest — SeasonalNaive

In [4]:
HORIZON    = 48   # 24-hour forecast
QUANTILES  = (0.1, 0.5, 0.9)

naive = SeasonalNaive(season=48)
result_naive = run_backtest(
    model=naive,
    bundle=bundle,
    origins=origins,
    horizon=HORIZON,
    quantiles=QUANTILES,
    refit=False,
    model_name='seasonal_naive',
)
print('SeasonalNaive backtest done. Valid origins:', len(result_naive.origins))

SeasonalNaive backtest done. Valid origins: 4


## Run backtest — GlobalLGBM

> Fitting LightGBM with sparse lags takes ~30–60 s on CPU (144 sub-models).  The cell below is not skipped — it is a feature of this project.

In [5]:
import warnings
warnings.filterwarnings('ignore')

lgbm = GlobalLGBM(
    quantiles=(0.1, 0.5, 0.9),
    lags=[-1, -2, -3, -48, -96],
    lags_past_covariates=[-1, -2, -3, -48],
    n_estimators=50,
    num_leaves=15,
    verbose=-1,
)

result_lgbm = run_backtest(
    model=lgbm,
    bundle=bundle,
    origins=origins,
    horizon=HORIZON,
    quantiles=QUANTILES,
    refit=False,
    model_name='global_lgbm',
)
print('GlobalLGBM backtest done. Valid origins:', len(result_lgbm.origins))

GlobalLGBM backtest done. Valid origins: 4


## Ablation table

In [6]:
results = {
    'seasonal_naive': result_naive,
    'global_lgbm':    result_lgbm,
}

# Optional: if Colab artefacts are present, load and add them here
# import joblib
# from src.models.global_dl import GlobalTiDE
# ...

ablation = build_ablation_table(results, baseline='seasonal_naive')
print('\n=== Ablation Table ===')
print(ablation.round(3).to_string())

# Save
Path('reports').mkdir(exist_ok=True)
ablation.to_csv('reports/ablation_table.csv')
print('\nSaved: reports/ablation_table.csv')
ablation.round(3)


=== Ablation Table ===
                pinball  coverage_80   crps    mae  skill_pinball  skill_mae
seasonal_naive    2.919        0.000  5.837  5.837          0.000      0.000
global_lgbm       1.446        0.703  2.893  4.068          0.504      0.303

Saved: reports/ablation_table.csv


,pinball,coverage_80,crps,mae,skill_pinball,skill_mae
seasonal_naive,2.919,0.000,5.837,5.837,0.000,0.000
global_lgbm,1.446,0.703,2.893,4.068,0.504,0.303


## Per-horizon MAE plot

In [7]:
def per_horizon_mae(result) -> np.ndarray:
    """MAE at each forecast step (averaged over origins)."""
    q_median = min(result.quantiles, key=lambda q: abs(q - 0.5))
    fc = result.forecasts[q_median]          # (n_origins, horizon)
    act = result.actuals                     # (n_origins, horizon)
    return np.mean(np.abs(act - fc), axis=0) # (horizon,)

mae_naive = per_horizon_mae(result_naive)
mae_lgbm  = per_horizon_mae(result_lgbm)

fig, ax = plt.subplots(figsize=(11, 3.5))
sp = np.arange(1, HORIZON + 1)
ax.plot(sp, mae_naive, label='SeasonalNaive', lw=1.5, color='grey', ls='--')
ax.plot(sp, mae_lgbm,  label='GlobalLGBM',   lw=1.5, color='royalblue')
ax.set_xlabel('Forecast step (settlement period offset)')
ax.set_ylabel('MAE (£/MWh)')
ax.set_title('Per-horizon MAE — fixture backtest')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('reports/figures/04_per_horizon_mae.png', dpi=120)
plt.show()
print('Saved: reports/figures/04_per_horizon_mae.png')

Saved: reports/figures/04_per_horizon_mae.png
